# 📊 Fairness Analysis of Loan Approval Decisions
## Keishana Morsley
## INSY 5378 - Special Topics in AI for Business

This notebook evaluates fairness in the provided `loan_dataset.xlsx` dataset using the same general workflow as the class fairness tutorial. The analysis focuses on:
- Disparate Impact (DI)
- Statistical Parity Difference (SPD)
- Equal Opportunity Difference (EOD)
- Average Odds Difference (AOD)

The protected demographic attributes evaluated are `Gender`, `Race`, and `Age`.

In [1]:
# Install necessary packages if not already installed
!pip install aif360 scikit-learn pandas openpyxl matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 5.8 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from aif360.datasets import StandardDataset
from aif360.metrics import ClassificationMetric

pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


### 📥 Load the Loan Dataset

In [3]:
file_name = "loan_dataset.xlsx"

df = pd.read_excel(file_name, sheet_name="Loan_Dataset")
data_dictionary = pd.read_excel(file_name, sheet_name="Data_Dictionary_&_Notes")

print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
df.head()

Dataset shape: (1500, 11)
Missing values: 0


,Applicant_ID,Age,Gender,Race,Annual_Income,Credit_Score,Loan_Amount_Requested,Debt_To_Income_Ratio,Employment_Length_Years,Savings_Balance,Loan_Approved
0,APP-00001,71,Female,White,71993,817,42001,0.44,2,20462,1
1,APP-00002,57,Male,Asian,75403,790,13007,0.46,13,26970,1
2,APP-00003,28,Male,Black,76444,631,40096,0.45,4,6499,1
3,APP-00004,27,Male,Black,57108,684,46787,0.26,4,974,0
4,APP-00005,34,Male,White,114106,658,14255,0.15,4,14615,1


### 🔎 Review the Protected Attributes

The data dictionary identifies `Gender` and `Race` as protected classes and specifically notes injected bias against Female applicants and historical penalties against Black and Hispanic applicants. `Age` is also included as a demographic attribute in the assignment, so it is audited using an operational split of **Under 40** versus **40+**.

In [4]:
df["Age_Group"] = np.where(df["Age"] < 40, "Under 40", "40+")

print("Gender counts:")
print(df["Gender"].value_counts())
print("\nRace counts:")
print(df["Race"].value_counts())
print("\nAge group counts:")
print(df["Age_Group"].value_counts())

Gender counts:
Gender
Female        723
Male          711
Non-Binary     66
Name: count, dtype: int64

Race counts:
Race
White       815
Black       305
Hispanic    227
Asian       153
Name: count, dtype: int64

Age group counts:
Age_Group
40+         939
Under 40    561
Name: count, dtype: int64


### 📈 Historical Approval Rates in the Original Data

In [5]:
gender_rates = df.groupby("Gender")["Loan_Approved"].agg(["count", "mean"])
race_rates = df.groupby("Race")["Loan_Approved"].agg(["count", "mean"])
age_rates = df.groupby("Age_Group")["Loan_Approved"].agg(["count", "mean"])

print("Approval rate by Gender:")
print(gender_rates.round(4))
print("\nApproval rate by Race:")
print(race_rates.round(4))
print("\nApproval rate by Age Group:")
print(age_rates.round(4))

Approval rate by Gender:
            count    mean
Gender                   
Female        723  0.5104
Male          711  0.5654
Non-Binary     66  0.5152

Approval rate by Race:
          count    mean
Race                   
Asian       153  0.6471
Black       305  0.3377
Hispanic    227  0.4053
White       815  0.6270

Approval rate by Age Group:
           count    mean
Age_Group               
40+          939  0.5527
Under 40     561  0.5098


### 🧮 Historical Disparate Impact and Statistical Parity

In [6]:
def historical_fairness(data, attribute, unprivileged_value, privileged_value):
    unpriv = data[data[attribute] == unprivileged_value]["Loan_Approved"].mean()
    priv = data[data[attribute] == privileged_value]["Loan_Approved"].mean()
    di = unpriv / priv
    spd = unpriv - priv
    return di, spd

historical_comparisons = [
    ("Gender: Female vs Male", "Gender", "Female", "Male"),
    ("Gender: Non-Binary vs Male", "Gender", "Non-Binary", "Male"),
    ("Race: Black vs White", "Race", "Black", "White"),
    ("Race: Hispanic vs White", "Race", "Hispanic", "White"),
    ("Race: Asian vs White", "Race", "Asian", "White"),
    ("Age: Under 40 vs 40+", "Age_Group", "Under 40", "40+")
]

for label, attribute, unprivileged, privileged in historical_comparisons:
    di, spd = historical_fairness(df, attribute, unprivileged, privileged)
    print(label)
    print("Disparate Impact:", round(di, 4))
    print("Statistical Parity Difference:", round(spd, 4))
    print()

Gender: Female vs Male
Disparate Impact: 0.9027
Statistical Parity Difference: -0.055

Gender: Non-Binary vs Male
Disparate Impact: 0.9111
Statistical Parity Difference: -0.0502

Race: Black vs White
Disparate Impact: 0.5386
Statistical Parity Difference: -0.2893

Race: Hispanic vs White
Disparate Impact: 0.6464
Statistical Parity Difference: -0.2217

Race: Asian vs White
Disparate Impact: 1.032
Statistical Parity Difference: 0.0201

Age: Under 40 vs 40+
Disparate Impact: 0.9224
Statistical Parity Difference: -0.0429



### ⚙️ Prepare the Data for Modeling

In [7]:
X = df.drop(["Applicant_ID", "Loan_Approved", "Age_Group"], axis=1)
y = df["Loan_Approved"]

# One-hot encode categorical features so each category is represented separately
X = pd.get_dummies(X, columns=["Gender", "Race"], drop_first=False, dtype=int)

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.3, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 1050
Testing rows: 450


### 🤖 Train a Logistic Regression Model

In [8]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_scaled, y_train)
y_pred = clf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.6844444444444444


### 📐 Create the Test Results Table

In [9]:
test_results = df.loc[X_test.index, ["Age", "Age_Group", "Gender", "Race", "Loan_Approved"]].copy()
test_results["predicted"] = y_pred

test_results.head()

,Age,Age_Group,Gender,Race,Loan_Approved,predicted
149,56,40+,Female,White,1,1
949,34,Under 40,Male,Black,0,0
1266,28,Under 40,Female,White,1,0
596,57,40+,Female,Black,0,0
881,32,Under 40,Female,Black,0,1


### ⚖️ Fairness Metrics Using AIF360

For each comparison below, the privileged/reference group is coded as `1` and the comparison group is coded as `0`, matching the structure used in the class tutorial.

- DI ideal ≈ 1
- SPD ideal ≈ 0
- EOD ideal ≈ 0
- AOD ideal ≈ 0

In [11]:
fairness_results = []

def evaluate_fairness(data, attribute, unprivileged_value, privileged_value, label):
    subset = data[data[attribute].isin([unprivileged_value, privileged_value])].copy()
    subset["protected"] = (subset[attribute] == privileged_value).astype(int)

    audit_df = subset[["Loan_Approved", "predicted", "protected"]].copy()

    test_dataset = StandardDataset(
        df=audit_df,
        label_name="Loan_Approved",
        favorable_classes=[1],
        protected_attribute_names=["protected"],
        privileged_classes=[[1]]
    )

    pred_dataset = test_dataset.copy()
    pred_dataset.labels = audit_df["predicted"].values.reshape(-1, 1)

    metric = ClassificationMetric(
        test_dataset,
        pred_dataset,
        unprivileged_groups=[{"protected": 0}],
        privileged_groups=[{"protected": 1}]
    )

    result = {
        "Comparison": label,
        "Disparate Impact": metric.disparate_impact(),
        "Statistical Parity Difference": metric.statistical_parity_difference(),
        "Equal Opportunity Difference": metric.equal_opportunity_difference(),
        "Average Odds Difference": metric.average_odds_difference(),
        "N": len(subset)
    }

    fairness_results.append(result)

    print(label)
    print("Disparate Impact:", result["Disparate Impact"])
    print("Statistical Parity Difference:", result["Statistical Parity Difference"])
    print("Equal Opportunity Difference:", result["Equal Opportunity Difference"])
    print("Average Odds Difference:", result["Average Odds Difference"])
    print("N:", result["N"])
    print()

    return result

### 👥 Gender Fairness

In [12]:
evaluate_fairness(test_results, "Gender", "Female", "Male", "Female vs Male")
evaluate_fairness(test_results, "Gender", "Non-Binary", "Male", "Non-Binary vs Male")

Female vs Male
Disparate Impact: 0.8163126361655774
Statistical Parity Difference: -0.11459395174991505
Equal Opportunity Difference: -0.12419637638807712
Average Odds Difference: -0.11209818819403855
N: 434

Non-Binary vs Male
Disparate Impact: 0.9016544117647058
Statistical Parity Difference: -0.06135321100917435
Equal Opportunity Difference: -0.08232445520581111
Average Odds Difference: -0.028940005380683337
N: 234



{'Comparison': 'Non-Binary vs Male',
 'Disparate Impact': np.float64(0.9016544117647058),
 'Statistical Parity Difference': np.float64(-0.06135321100917435),
 'Equal Opportunity Difference': np.float64(-0.08232445520581111),
 'Average Odds Difference': np.float64(-0.028940005380683337),
 'N': 234}

### 🌎 Race Fairness

In [13]:
evaluate_fairness(test_results, "Race", "Black", "White", "Black vs White")
evaluate_fairness(test_results, "Race", "Hispanic", "White", "Hispanic vs White")
evaluate_fairness(test_results, "Race", "Asian", "White", "Asian vs White")

Black vs White
Disparate Impact: 0.1459513164431197
Statistical Parity Difference: -0.6915526950925182
Equal Opportunity Difference: -0.7076813655761024
Average Odds Difference: -0.6390970930444615
N: 336

Hispanic vs White
Disparate Impact: 0.32216678545972915
Statistical Parity Difference: -0.548864948056945
Equal Opportunity Difference: -0.5168918918918919
Average Odds Difference: -0.48536902286902284
N: 295

Asian vs White
Disparate Impact: 1.1251973284760168
Statistical Parity Difference: 0.10137659783677477
Equal Opportunity Difference: 0.043591979075849996
Average Odds Difference: 0.12344434118627662
N: 271



{'Comparison': 'Asian vs White',
 'Disparate Impact': np.float64(1.1251973284760168),
 'Statistical Parity Difference': np.float64(0.10137659783677477),
 'Equal Opportunity Difference': np.float64(0.043591979075849996),
 'Average Odds Difference': np.float64(0.12344434118627662),
 'N': 271}

### 🎂 Age Fairness

In [14]:
evaluate_fairness(test_results, "Age_Group", "Under 40", "40+", "Under 40 vs 40+")

Under 40 vs 40+
Disparate Impact: 0.8214285714285714
Statistical Parity Difference: -0.10869565217391308
Equal Opportunity Difference: -0.1007872846108141
Average Odds Difference: -0.09085497715131363
N: 450



{'Comparison': 'Under 40 vs 40+',
 'Disparate Impact': np.float64(0.8214285714285714),
 'Statistical Parity Difference': np.float64(-0.10869565217391308),
 'Equal Opportunity Difference': np.float64(-0.1007872846108141),
 'Average Odds Difference': np.float64(-0.09085497715131363),
 'N': 450}

### 📋 Summary of Model Fairness Metrics

In [15]:
fairness_summary = pd.DataFrame(fairness_results)
fairness_summary.round(4)

,Comparison,Disparate Impact,Statistical Parity Difference,Equal Opportunity Difference,Average Odds Difference,N
0,Female vs Male,0.8163,-0.1146,-0.1242,-0.1121,434
1,Non-Binary vs Male,0.9017,-0.0614,-0.0823,-0.0289,234
2,Black vs White,0.1460,-0.6916,-0.7077,-0.6391,336
3,Hispanic vs White,0.3222,-0.5489,-0.5169,-0.4854,295
4,Asian vs White,1.1252,0.1014,0.0436,0.1234,271
5,Under 40 vs 40+,0.8214,-0.1087,-0.1008,-0.0909,450


## 🧠 Conclusions

The fairness audit shows that the dataset contains meaningful demographic disparities, especially by race.

1. **Race shows the strongest evidence of bias in the historical decisions.** Black applicants have a historical approval rate of approximately **33.8%**, and Hispanic applicants approximately **40.5%**, compared with approximately **62.7%** for White applicants. Their historical disparate-impact ratios are approximately **0.54** and **0.65**, respectively. These results are consistent with the data dictionary, which states that bias was embedded against Black and Hispanic applicants.

2. **The trained model reproduces and amplifies the racial disparities present in the historical labels.** On the test set, Black applicants have a DI of approximately **0.146** versus White applicants, with SPD ≈ **-0.692**, EOD ≈ **-0.708**, and AOD ≈ **-0.639**. Hispanic applicants also show substantial disparities (DI ≈ **0.322**, SPD ≈ **-0.549**). These values are far from the ideal fairness values of DI ≈ 1 and the difference-based metrics ≈ 0.

3. **Gender disparity is present but smaller than the race disparity.** Female applicants have a historical approval rate of approximately **51.0%** versus **56.5%** for Male applicants. In model predictions, Female vs Male DI is approximately **0.816** and SPD approximately **-0.115**. The negative EOD and AOD also indicate lower favorable error-rate outcomes for Female applicants.

4. **The Non-Binary comparison is closer to parity, but the group is small.** Only 66 Non-Binary applicants appear in the full dataset and 16 appear in the test set, so the model-based fairness estimates for this group should be interpreted cautiously.

5. **The age comparison shows a modest disparity.** Applicants under 40 have a lower approval rate than applicants 40+ in both historical labels and model predictions, although the disparity is much smaller than the Black/White and Hispanic/White differences. The 40-year split is used only as an operational audit grouping and should not be interpreted as a legal determination of privilege.

### Overall Conclusion
The data are **not demographically neutral**. The clearest bias is against **Black and Hispanic applicants**, and the predictive model learns these patterns from the historical target label. Because protected attributes and biased historical outcomes are included in the modeling process, a model can achieve reasonable accuracy while still producing unfair outcomes. Fairness metrics therefore provide information that accuracy alone does not capture.